# Data Understanding

*Generated by DoML — Do Machine Learning*

In [ ]:
# Reproducibility — REPR-01 (non-negotiable per CLAUDE.md)
set.seed(42)

In [ ]:
library(jsonlite)
library(duckdb)

# Project root — REPR-02 (non-negotiable per CLAUDE.md)
PROJECT_ROOT <- Sys.getenv("PROJECT_ROOT", unset = "/home/jovyan/work")
cat("Project root:", PROJECT_ROOT, "\n")

config_path <- file.path(PROJECT_ROOT, ".planning", "config.json")
config <- fromJSON(config_path)

cat("Problem type:", config$problem_type, "\n")
cat("Time factor: ", isTRUE(config$time_factor), "\n")
cat("Language:    ", config$language, "\n")

## 1. DuckDB Data Profiling

Zero-copy scan of `data/raw/` — schema, row counts, null counts, numeric distributions, and categorical unique counts via SQL (EDA-02).

In [ ]:
library(tidyverse)

data_dir <- file.path(PROJECT_ROOT, "data", "raw")
supported_ext <- c("csv", "parquet", "xlsx")
all_files <- list.files(data_dir, recursive = TRUE, full.names = TRUE)
files <- all_files[tools::file_ext(all_files) %in% supported_ext]

if (length(files) == 0) {
  cat("No supported files found in", data_dir, "\n")
  cat("Supported: .csv, .parquet, .xlsx\n")
} else {
  cat("Found", length(files), "file(s):\n")
  for (f in files) cat(" ", basename(f), "\n")
}

get_read_fn <- function(p, ext) {
  switch(ext,
    "csv"     = paste0("read_csv_auto('", p, "')"),
    "parquet" = paste0("read_parquet('", p, "')"),
    "xlsx"    = paste0("read_xlsx('", p, "')"),
    stop(paste("Unsupported extension:", ext))
  )
}

for (fpath in files) {
  con <- dbConnect(duckdb())
  ext  <- tools::file_ext(fpath)
  rfn  <- get_read_fn(fpath, ext)

  schema    <- dbGetQuery(con, paste("DESCRIBE SELECT * FROM", rfn))
  row_count <- dbGetQuery(con, paste("SELECT COUNT(*) AS n FROM", rfn))$n
  cols      <- schema$column_name

  null_exprs <- paste(
    sprintf("SUM(CASE WHEN \"%s\" IS NULL THEN 1 ELSE 0 END) AS \"%s\"", cols, cols),
    collapse = ", "
  )
  null_row <- dbGetQuery(con, paste("SELECT", null_exprs, "FROM", rfn))
  null_df <- data.frame(
    column     = cols,
    null_count = as.integer(null_row[1, ]),
    null_pct   = round(as.numeric(null_row[1, ]) / row_count * 100, 1)
  )
  dbDisconnect(con)

  cat("\n###", basename(fpath), "\n")
  cat("Shape:", format(row_count, big.mark = ","), "rows x", length(cols), "columns\n")
  cat("\nSchema:\n")
  print(schema[, c("column_name", "column_type")])

  has_nulls <- null_df[null_df$null_count > 0, ]
  if (nrow(has_nulls) > 0) {
    cat("\nNull counts:\n")
    print(has_nulls)
  } else {
    cat("\u2713 No null values\n")
  }
}

In [ ]:
numeric_types <- c("INTEGER", "BIGINT", "DOUBLE", "FLOAT", "HUGEINT", "SMALLINT",
                   "TINYINT", "DECIMAL", "NUMERIC", "REAL")

for (fpath in files) {
  con <- dbConnect(duckdb())
  ext <- tools::file_ext(fpath)
  rfn <- get_read_fn(fpath, ext)

  schema   <- dbGetQuery(con, paste("DESCRIBE SELECT * FROM", rfn))
  num_cols <- schema$column_name[
    sapply(schema$column_type, function(t)
      any(sapply(numeric_types, function(nt) grepl(nt, t, ignore.case = TRUE))))
  ]

  if (length(num_cols) == 0) {
    cat(basename(fpath), ": No numeric columns\n")
    dbDisconnect(con)
    next
  }

  stat_exprs <- paste(
    sprintf("MIN(\"%s\") AS \"%s__min\", MAX(\"%s\") AS \"%s__max\", AVG(\"%s\") AS \"%s__avg\", STDDEV(\"%s\") AS \"%s__std\", MEDIAN(\"%s\") AS \"%s__med\"",
            num_cols, num_cols, num_cols, num_cols, num_cols,
            num_cols, num_cols, num_cols, num_cols, num_cols),
    collapse = ", "
  )
  stats_wide <- dbGetQuery(con, paste("SELECT", stat_exprs, "FROM", rfn))
  dbDisconnect(con)

  stats_df <- data.frame(
    column = num_cols,
    min    = round(as.numeric(stats_wide[1, paste0(num_cols, "__min")]), 4),
    max    = round(as.numeric(stats_wide[1, paste0(num_cols, "__max")]), 4),
    mean   = round(as.numeric(stats_wide[1, paste0(num_cols, "__avg")]), 4),
    std    = round(as.numeric(stats_wide[1, paste0(num_cols, "__std")]), 4),
    median = round(as.numeric(stats_wide[1, paste0(num_cols, "__med")]), 4)
  )
  cat("\n###", basename(fpath), "\u2014 Numeric Distributions (DuckDB)\n")
  print(stats_df)
}

In [ ]:
for (fpath in files) {
  con     <- dbConnect(duckdb())
  ext     <- tools::file_ext(fpath)
  rfn     <- get_read_fn(fpath, ext)
  schema  <- dbGetQuery(con, paste("DESCRIBE SELECT * FROM", rfn))
  row_cnt <- dbGetQuery(con, paste("SELECT COUNT(*) AS n FROM", rfn))$n

  str_cols <- schema$column_name[grepl("VARCHAR|TEXT", schema$column_type, ignore.case = TRUE)]

  if (length(str_cols) == 0) {
    cat(basename(fpath), ": No categorical columns\n")
    dbDisconnect(con)
    next
  }

  results <- lapply(str_cols, function(c) {
    uniq <- dbGetQuery(con, sprintf("SELECT COUNT(DISTINCT \"%s\") AS n FROM %s", c, rfn))$n
    data.frame(column = c, unique_count = uniq, cardinality_pct = round(uniq / row_cnt * 100, 1))
  })
  uniq_df <- do.call(rbind, results)

  cat("\n###", basename(fpath), "\u2014 Categorical Columns\n")
  print(uniq_df)

  low_card <- uniq_df$column[uniq_df$cardinality_pct < 5.0]
  for (c in head(low_card, 5)) {
    top <- dbGetQuery(con, sprintf(
      "SELECT \"%s\", COUNT(*) AS freq FROM %s GROUP BY \"%s\" ORDER BY freq DESC LIMIT 10", c, rfn, c
    ))
    cat("\nTop values \u2014", c, ":\n")
    print(top)
  }
  dbDisconnect(con)
}

## 2. Load Data for Statistical Analysis

Loading dataset into a data frame for distribution analysis, correlations, and visualizations.

**Sampling rule (Decision 5):** If dataset exceeds 500,000 rows, a 50,000-row sample is used for statistical computations.

In [ ]:
SAMPLE_THRESHOLD <- 500000L
dfs <- list()

for (fpath in files) {
  con <- dbConnect(duckdb())
  ext <- tools::file_ext(fpath)
  rfn <- get_read_fn(fpath, ext)

  row_cnt <- dbGetQuery(con, paste("SELECT COUNT(*) AS n FROM", rfn))$n

  if (row_cnt > SAMPLE_THRESHOLD) {
    df <- dbGetQuery(con, paste("SELECT * FROM", rfn, "USING SAMPLE 50000"))
    cat(basename(fpath), ": Loaded 50,000-row sample (full dataset:", format(row_cnt, big.mark=","), "rows)\n")
  } else {
    df <- dbGetQuery(con, paste("SELECT * FROM", rfn))
    cat(basename(fpath), ": Loaded full dataset (", format(row_cnt, big.mark=","), "rows x", ncol(df), "columns)\n")
  }
  dfs[[basename(fpath)]] <- df
  dbDisconnect(con)
}

## 3. Distribution Analysis

Histograms, Q-Q plots, Shapiro-Wilk normality test, skewness, and kurtosis for all numeric features (EDA-03).

In [ ]:
library(ggplot2)

for (fname in names(dfs)) {
  df <- dfs[[fname]]
  num_cols <- names(df)[sapply(df, is.numeric)]

  if (length(num_cols) == 0) {
    cat(fname, ": No numeric columns \u2014 skipping\n")
    next
  }

  # Histograms via ggplot2
  df_long <- df %>%
    select(all_of(num_cols)) %>%
    pivot_longer(everything(), names_to = "variable", values_to = "value")
  p <- ggplot(df_long, aes(x = value)) +
    geom_histogram(bins = 30, fill = "steelblue", color = "white", alpha = 0.8) +
    facet_wrap(~variable, scales = "free") +
    labs(title = paste(fname, "\u2014 Numeric Distributions"), x = NULL, y = "Frequency") +
    theme_minimal()
  print(p)

  # Q-Q plots + Shapiro-Wilk
  cat("\n###", fname, "\u2014 Normality Tests\n")
  results <- lapply(num_cols, function(col) {
    x <- na.omit(df[[col]])
    n <- length(x)
    if (n < 3) return(NULL)

    qqnorm(x, main = paste("Q-Q:", col), col = "steelblue", pch = 16, cex = 0.5)
    qqline(x, col = "red", lwd = 2)

    # Shapiro-Wilk (R max n=5000 — sample if larger)
    x_test <- if (n > 5000) sample(x, 5000) else x
    sw <- shapiro.test(x_test)

    data.frame(
      column    = col,
      n         = n,
      test      = "Shapiro-Wilk",
      statistic = round(sw$statistic, 4),
      p_value   = round(sw$p.value, 4),
      is_normal = sw$p.value > 0.05,
      skewness  = round(mean((x - mean(x))^3) / sd(x)^3, 3),
      kurtosis  = round(mean((x - mean(x))^4) / sd(x)^4 - 3, 3)
    )
  })
  results_df <- do.call(rbind, Filter(Negate(is.null), results))
  if (!is.null(results_df)) print(results_df)
}

## 4. Correlation Analysis

Method selected per feature type (EDA-04):
- **Both numeric** \u2192 Pearson correlation
- **Both categorical** \u2192 Cram\u00e9r's V
- **One numeric + one binary** \u2192 point-biserial

In [ ]:
# Note: corrplot is not available in this environment — using ggplot2 heatmap instead.
# To install: install.packages('corrplot') and rebuild the Docker image.

cramers_v <- function(x, y) {
  ct    <- table(x, y)
  chi2  <- chisq.test(ct, correct = FALSE)$statistic
  n     <- sum(ct)
  phi2  <- chi2 / n
  r     <- nrow(ct); k <- ncol(ct)
  denom <- min(k - 1, r - 1)
  if (denom == 0) return(0)
  sqrt(phi2 / denom)
}

is_categorical <- function(x, threshold = 0.1) {
  is.character(x) || is.factor(x) || (length(unique(x)) < max(10, length(x) * threshold))
}

for (fname in names(dfs)) {
  df <- dfs[[fname]]
  cat("\n###", fname, "\u2014 Correlation Analysis\n")

  # Pearson correlation matrix
  num_df <- df[, sapply(df, is.numeric), drop = FALSE]
  if (ncol(num_df) >= 2) {
    corr_mat <- cor(num_df, use = "pairwise.complete.obs", method = "pearson")
    cat("\nPearson Correlation:\n")
    print(round(corr_mat, 3))

    # ggplot2 heatmap (corrplot not available)
    corr_df <- as.data.frame(corr_mat) %>%
      rownames_to_column("row") %>%
      pivot_longer(-row, names_to = "col", values_to = "correlation")
    p <- ggplot(corr_df, aes(x = col, y = row, fill = correlation)) +
      geom_tile(color = "white") +
      geom_text(aes(label = round(correlation, 2)), size = 3) +
      scale_fill_gradient2(low = "#2166AC", mid = "white", high = "#D6604D",
                            midpoint = 0, limits = c(-1, 1)) +
      labs(title = paste(fname, "\u2014 Pearson Correlation Heatmap"),
           x = NULL, y = NULL, fill = "r") +
      theme_minimal() +
      theme(axis.text.x = element_text(angle = 45, hjust = 1))
    print(p)
  }

  # Mixed-type correlations
  cols <- names(df)
  mixed_results <- list()
  for (i in seq_along(cols)) {
    for (j in seq(i + 1, length(cols))) {
      if (j > length(cols)) next
      c1 <- cols[i]; c2 <- cols[j]
      common_idx <- intersect(which(!is.na(df[[c1]])), which(!is.na(df[[c2]])))
      if (length(common_idx) < 10) next
      x <- df[[c1]][common_idx]; y <- df[[c2]][common_idx]

      cat1 <- is_categorical(x); cat2 <- is_categorical(y)
      bin1 <- length(unique(x)) == 2; bin2 <- length(unique(y)) == 2

      if (cat1 && cat2) {
        v <- cramers_v(as.character(x), as.character(y))
        mixed_results[[length(mixed_results)+1]] <- data.frame(col1=c1, col2=c2, method="Cram\u00e9r's V", value=round(v,3))
      } else if (!cat1 && bin2) {
        r_val <- cor(as.numeric(y), as.numeric(x), use="complete.obs")
        mixed_results[[length(mixed_results)+1]] <- data.frame(col1=c1, col2=c2, method="point-biserial", value=round(r_val,3))
      } else if (bin1 && !cat2) {
        r_val <- cor(as.numeric(x), as.numeric(y), use="complete.obs")
        mixed_results[[length(mixed_results)+1]] <- data.frame(col1=c1, col2=c2, method="point-biserial", value=round(r_val,3))
      }
    }
  }
  if (length(mixed_results) > 0) {
    cat("\nMixed-type correlations:\n")
    print(do.call(rbind, mixed_results))
  } else {
    cat("No mixed-type column pairs detected (or all numeric \u2014 see Pearson matrix above)\n")
  }
}

## 5. Missing Value Analysis

Missingness heatmap, percentage by feature, and correlation of missingness (EDA-05).

In [ ]:
for (fname in names(dfs)) {
  df <- dfs[[fname]]
  null_mask <- is.na(df)
  total_missing <- sum(null_mask)

  if (total_missing == 0) {
    cat(fname, ": No missing values \u2014 skipping missingness plots\n")
    next
  }

  cat("\n###", fname, "\u2014 Missing Value Analysis\n")
  cat("Total missing:", format(total_missing, big.mark=","),
      sprintf("(%.1f%% of all cells)\n", total_missing / (nrow(df)*ncol(df)) * 100))

  # Percentage missing bar chart
  pct_miss <- colMeans(null_mask) * 100
  pct_df <- data.frame(
    column      = names(pct_miss),
    pct_missing = pct_miss
  ) %>% filter(pct_missing > 0) %>% arrange(pct_missing)

  p <- ggplot(pct_df, aes(x = reorder(column, pct_missing), y = pct_missing)) +
    geom_col(fill = "steelblue") +
    geom_hline(yintercept = 5,  color = "orange", linetype = "dashed") +
    geom_hline(yintercept = 30, color = "red",    linetype = "dashed") +
    coord_flip() +
    labs(title = paste(fname, "\u2014 % Missing by Feature"), x = NULL, y = "% Missing") +
    theme_minimal()
  print(p)

  # Missingness heatmap (ggplot2 geom_tile)
  null_long <- as.data.frame(null_mask) %>%
    mutate(row_id = row_number()) %>%
    pivot_longer(-row_id, names_to = "column", values_to = "is_missing")
  p2 <- ggplot(null_long, aes(x = row_id, y = column, fill = is_missing)) +
    geom_tile() +
    scale_fill_manual(values = c("FALSE" = "grey90", "TRUE" = "gold"),
                      labels = c("present", "missing")) +
    labs(title = paste(fname, "\u2014 Missingness Heatmap"),
         x = "Observation", y = NULL, fill = NULL) +
    theme_minimal() +
    theme(axis.text.x = element_blank())
  print(p2)

  # Correlation of missingness
  miss_cols <- names(which(colSums(null_mask) > 0))
  if (length(miss_cols) >= 2) {
    miss_corr <- cor(null_mask[, miss_cols, drop = FALSE] * 1.0)
    cat("\nMissingness correlation:\n")
    print(round(miss_corr, 2))

    # ggplot2 heatmap for missingness correlation
    mc_df <- as.data.frame(miss_corr) %>%
      rownames_to_column("row") %>%
      pivot_longer(-row, names_to = "col", values_to = "correlation")
    p3 <- ggplot(mc_df, aes(x = col, y = row, fill = correlation)) +
      geom_tile(color = "white") +
      geom_text(aes(label = round(correlation, 2)), size = 3) +
      scale_fill_gradient2(low = "#2166AC", mid = "white", high = "#D6604D",
                            midpoint = 0, limits = c(-1, 1)) +
      labs(title = paste(fname, "\u2014 Missingness Correlation"),
           x = NULL, y = NULL, fill = "r") +
      theme_minimal() +
      theme(axis.text.x = element_text(angle = 45, hjust = 1))
    print(p3)
  }
}

### MCAR / MAR / MNAR Assessment

Use the heatmap and correlation matrix above to assess the missingness mechanism:

| Mechanism | What it means | Visual signal | Safe action |
|-----------|---------------|---------------|-------------|
| **MCAR** \u2014 Missing Completely At Random | Missingness unrelated to any variable | Heatmap shows random scatter; correlation near zero | Complete-case analysis valid |
| **MAR** \u2014 Missing At Random | Missingness related to **observed** variables | Non-zero correlations between missing indicators | Multiple imputation using correlated columns |
| **MNAR** \u2014 Missing Not At Random | Missingness related to the **missing value itself** | Cannot detect statistically | Domain investigation required |

**Analyst action:** Non-zero correlations in the missingness matrix are evidence of MAR (at minimum). Investigate whether those columns share a data-generating process.

## 6. Time Series Analysis

*Activated when `time_factor = true` in `.planning/config.json`.*

ADF and KPSS stationarity tests, and seasonal decomposition (EDA-06).

In [ ]:
library(tseries)

if (!isTRUE(config$time_factor)) {
  cat("time_factor = FALSE \u2014 time series analysis skipped.\n")
  cat("To enable: set time_factor=true in .planning/config.json\n")
} else {
  cat("### Time Series Stationarity & Decomposition\n")

  for (fname in names(dfs)) {
    df <- dfs[[fname]]
    num_cols <- names(df)[sapply(df, is.numeric)]
    if (length(num_cols) == 0) {
      cat(fname, ": No numeric columns\n")
      next
    }

    cat("\n####", fname, "\n")

    for (col in head(num_cols, 2)) {
      x <- na.omit(df[[col]])
      if (length(x) < 20) {
        cat(" ", col, ": Too few observations (n =", length(x), ") \u2014 skipping\n")
        next
      }

      cat("\n**", col, "(n =", length(x), ")**\n")

      # ADF test (H0: unit root = non-stationary)
      tryCatch({
        adf_res <- adf.test(x)
        cat("ADF Test  \u2014 statistic:", round(adf_res$statistic, 4),
            " p-value:", round(adf_res$p.value, 4), "\n")
        cat("           \u2192", if (adf_res$p.value < 0.05) "STATIONARY" else "NON-STATIONARY", "\n")
      }, error = function(e) cat("  ADF failed:", conditionMessage(e), "\n"))

      # KPSS test (H0: stationary)
      tryCatch({
        kp_res <- kpss.test(x)
        cat("KPSS Test \u2014 statistic:", round(kp_res$statistic, 4),
            " p-value:", round(kp_res$p.value, 4), "\n")
        cat("           \u2192", if (kp_res$p.value < 0.05) "NON-STATIONARY" else "STATIONARY", "\n")
      }, error = function(e) cat("  KPSS failed:", conditionMessage(e), "\n"))

      # Seasonal decomposition
      tryCatch({
        period <- 12L
        if (length(x) >= 2 * period) {
          ts_obj <- ts(x, frequency = period)
          decomp <- decompose(ts_obj, type = "additive")
          plot(decomp, col = c("steelblue", "darkorange", "green4", "gray50"))
          title(paste(col, "\u2014 Seasonal Decomposition (period =", period, ")"),
                outer = TRUE, line = -1)
        } else {
          cat("  Decomposition skipped: need >=", 2 * period, "obs (have", length(x), ")\n")
        }
      }, error = function(e) cat("  Decomposition failed:", conditionMessage(e), "\n"))
    }
  }
}

## 7. Tidy Data Validation

Validates data against tidy principles before feature engineering (EDA-07).

Checks:
1. Duplicate rows
2. Multiple values in one cell (comma/semicolon/pipe-separated strings)
3. Mixed observation types (column-name prefix heuristic)

Violations are **flagged only** \u2014 not silently fixed.

In [ ]:
for (fname in names(dfs)) {
  df <- dfs[[fname]]
  cat("\n###", fname, "\u2014 Tidy Validation\n")
  issues <- character(0)

  # Check 1: Duplicate rows
  dup_count <- sum(duplicated(df))
  if (dup_count > 0) {
    cat("[VIOLATION] Duplicate rows:", dup_count, "exact duplicates\n")
    print(head(df[duplicated(df) | duplicated(df, fromLast = TRUE), ], 3))
    issues <- c(issues, paste0("Duplicate rows (", dup_count, ") \u2014 action: df[!duplicated(df),]"))
  } else {
    cat("\u2713 No duplicate rows\n")
  }

  # Check 2: Multi-value cells (>5% contain comma/semicolon/pipe)
  char_cols <- names(df)[sapply(df, is.character)]
  multi_cols <- Filter(Negate(is.null), lapply(char_cols, function(col) {
    non_null <- na.omit(df[[col]])
    if (length(non_null) == 0) return(NULL)
    pct <- mean(grepl("[,;|]", non_null))
    if (pct > 0.05) data.frame(column = col, pct_multi_value = round(pct * 100, 1))
    else NULL
  }))
  if (length(multi_cols) > 0) {
    cat("[VIOLATION] Multi-value cells (>5% contain ,;|):\n")
    print(do.call(rbind, multi_cols))
    issues <- c(issues, paste0("Multi-value columns: ",
                               paste(sapply(multi_cols, `[[`, "column"), collapse=", ")))
  } else {
    cat("\u2713 No multi-value cells\n")
  }

  # Check 3: Mixed entity types (column prefix heuristic)
  prefixes <- table(sapply(names(df), function(n) {
    parts <- strsplit(n, "_")[[1]]
    if (length(parts) > 1) tolower(parts[1]) else tolower(substr(n, 1, 4))
  }))
  dominant <- names(prefixes[prefixes >= 2])
  if (length(dominant) >= 3) {
    cat("[POTENTIAL VIOLATION] Mixed entity types:", length(dominant),
        "column-name prefixes suggest multiple entities:",
        paste(head(dominant, 5), collapse=", "), "\n")
    issues <- c(issues, paste0("Mixed entity prefixes: ",
                               paste(head(dominant, 5), collapse=", ")))
  } else {
    cat("\u2713 No obvious mixed entity types\n")
  }

  if (length(issues) > 0) {
    cat("\nTidy violations summary \u2014 remediate before feature engineering:\n")
    for (iss in issues) cat(" \u2022", iss, "\n")
  } else {
    cat("\n\u2713", fname, ": All tidy checks passed\n")
  }
}

## 8. Processed Dataset

Structural cleaning only \u2014 writes output to `data/processed/` (EDA-08).

**Applied:**
- Drop fully duplicate rows
- Parse unambiguous date string columns

**Deferred to Phase 6 (Modelling):**
- Imputation \u2014 belongs inside R `recipes` pipeline
- Feature engineering \u2014 EDA identifies candidates; Modelling implements them

In [ ]:
processed_dir <- file.path(PROJECT_ROOT, "data", "processed")
dir.create(processed_dir, showWarnings = FALSE, recursive = TRUE)

for (fpath in files) {
  con     <- dbConnect(duckdb())
  ext     <- tools::file_ext(fpath)
  rfn     <- get_read_fn(fpath, ext)
  df_orig <- dbGetQuery(con, paste("SELECT * FROM", rfn))
  dbDisconnect(con)

  rows_before <- nrow(df_orig)
  df_clean    <- df_orig[!duplicated(df_orig), ]

  # Parse unambiguous date string columns
  date_cols <- character(0)
  for (col in names(df_clean)[sapply(df_clean, is.character)]) {
    sample_vals <- na.omit(df_clean[[col]])[seq_len(min(20, nrow(df_clean)))]
    parsed <- tryCatch(as.Date(sample_vals), error = function(e) NULL)
    if (!is.null(parsed) && all(!is.na(parsed))) {
      df_clean[[col]] <- as.Date(df_clean[[col]])
      date_cols <- c(date_cols, col)
    }
  }

  out_path <- file.path(processed_dir, basename(fpath))
  if (ext == "csv") {
    write.csv(df_clean, out_path, row.names = FALSE)
  } else if (ext == "parquet") {
    tryCatch(
      arrow::write_parquet(df_clean, out_path),
      error = function(e) {
        cat("  arrow not available \u2014 writing CSV fallback\n")
        write.csv(df_clean, sub("\\.parquet$", ".csv", out_path), row.names = FALSE)
      }
    )
  } else if (ext == "xlsx") {
    tryCatch(
      writexl::write_xlsx(df_clean, out_path),
      error = function(e) write.csv(df_clean, sub("\\.xlsx$", ".csv", out_path), row.names = FALSE)
    )
  }

  cat(basename(fpath), ":\n")
  cat("  Input: ", format(rows_before, big.mark=","), "rows x", ncol(df_orig), "columns\n")
  cat("  Output:", format(nrow(df_clean), big.mark=","), "rows (dropped",
      rows_before - nrow(df_clean), "duplicates)\n")
  if (length(date_cols) > 0) cat("  Parsed date columns:", paste(date_cols, collapse=", "), "\n")
  cat("  Written:", out_path, "\n\n")
}

## Caveats

*Assumptions and limitations for this EDA:*

- **Correlation is not causation** \u2014 all findings in this notebook are observational. Do not infer causal relationships without a controlled experiment.
- Distribution and correlation analysis was computed on the loaded dataset (see Data Loading section for sampling details if dataset exceeded 500,000 rows).
- Tidy violations are flagged but not remediated in this phase \u2014 review and fix before feature engineering.
- Imputation is deferred to Phase 6 (Modelling), embedded in an R `recipes` pipeline appropriate to the chosen model.
- Time series stationarity tests are only activated when `time_factor = TRUE` in `.planning/config.json`.
- The processed dataset in `data/processed/` reflects structural changes only. Raw data in `data/raw/` is unchanged (INFR-05).